In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu google-genai gradio

In [ ]:
import os
import re
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

from google import genai
from google.genai import types

import gradio as gr

print("All libraries imported successfully!")

In [ ]:
from google.colab import files

pdf_files = []

while len(pdf_files) < 5:
    print(f"Please upload {5 - len(pdf_files)} more PDF file(s).")

    uploaded = files.upload()

    for file in uploaded.keys():
        if file.lower().endswith(".pdf"):
            pdf_files.append(file)
        else:
            print(f"Skipped: {file} (not a PDF)")

    # Remove duplicates
    pdf_files = list(dict.fromkeys(pdf_files))

print("\nUploaded 5 PDF files:")
for file in pdf_files:
    print("-", file)

In [ ]:
documents = []

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text:
            text = text.strip()

            documents.append({
                "text": text,
                "source": pdf_file,
                "page": page_number
            })

print("Total pages extracted:", len(documents))

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Text cleaning completed!")

In [ ]:
def create_chunks(text, chunk_size=800, overlap=150):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [ ]:
def get_policy_type(filename):
    name = filename.lower()

    if "leave" in name:
        return "Leave Policy"
    elif "attendance" in name:
        return "Attendance Policy"
    elif "wfh" in name or "work_from_home" in name or "remote" in name:
        return "Work From Home Policy"
    elif "reimbursement" in name:
        return "Reimbursement Policy"
    elif "handbook" in name:
        return "Employee Handbook"
    else:
        return "General Policy"


chunks = []

for doc in documents:
    text_chunks = create_chunks(doc["text"])

    for chunk in text_chunks:
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "policy_type": get_policy_type(doc["source"])
        })

print("Total chunks:", len(chunks))

In [ ]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("Chunk ID:", i)
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Policy Type:", chunk["policy_type"])
    print("Text:", chunk["text"][:500])

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

In [ ]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

print("Embeddings normalized!")

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS vector database created!")
print("Number of vectors:", index.ntotal)

In [ ]:
def retrieve_documents(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            result = chunks[idx].copy()
            result["score"] = float(score)
            results.append(result)

    return results

In [ ]:
query = "How many casual leaves are allowed?"

results = retrieve_documents(query, top_k=5)

for result in results:
    print("=" * 80)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Policy:", result["policy_type"])
    print("Text:", result["text"][:500])

In [ ]:
!pip install -q transformers sentencepiece accelerate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 model loaded successfully!")

In [ ]:
SYSTEM_PROMPT = """
You are a Company Policy Assistant.

Your task is to answer employee questions ONLY using the
information provided in the retrieved company policy documents.

Rules:

1. Use only the provided context.
2. Do not use outside knowledge.
3. Do not invent or assume policy information.
4. If the answer is not present in the context, say:
   "Information not found in the provided company policy documents."
5. Give a clear and concise answer.
6. Mention the relevant policy when possible.
7. Do not make up document names or page numbers.
"""

In [ ]:
def build_context(results):
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"""
SOURCE {i}
Document: {result['source']}
Page: {result['page']}
Policy Type: {result['policy_type']}

Content:
{result['text']}
"""
        )

    return "\n".join(context_parts)

In [ ]:
def rag_answer(query, top_k=3):

    results = retrieve_documents(query, top_k=top_k)

    if not results:
        return (
            "Information not found in the provided company policy documents.",
            []
        )

    best_score = results[0]["score"]

    if best_score < 0.15:
        return (
            "Information not found in the provided company policy documents.",
            []
        )

    context = build_context(results)

    prompt = f"""
You are a company policy question-answering assistant.

Answer the employee's question using ONLY the information in the
provided company policy context.

IMPORTANT:
- Give a complete sentence.
- Give the exact number or rule when available.
- Do not add information that is not present in the context.
- If the answer is not present in the context, say:
Information not found in the provided company policy documents.

Company Policy Context:
{context}

Employee Question:
{query}

Answer in one clear complete sentence:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=50,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, results

In [ ]:
question = "How many casual leaves are allowed?"

answer, sources = rag_answer(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")

for source in sources[:3]:
    print(
        f"- {source['source']} | "
        f"Page {source['page']} | "
        f"{source['policy_type']}"
    )

In [ ]:
question = "What is the capital of France?"

answer, sources = rag_answer(question)

print(answer)

In [ ]:
test_questions = [
    "How many casual leaves are allowed?",
    "What is the work from home policy?",
    "What is the attendance requirement?",
    "What are the reimbursement rules?",
    "How many sick leaves can an employee take?"
]

for question in test_questions:

    print("=" * 80)
    print("QUESTION:", question)

    answer, sources = rag_answer(question)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")

    for source in sources[:2]:
        print(
            f"{source['source']} - Page {source['page']}"
        )

    print()

In [ ]:
def format_sources(sources):
    if not sources:
        return "No relevant source found."

    source_text = "\n\n### Sources\n"

    seen = set()

    for source in sources:
        key = (source["source"], source["page"])

        if key not in seen:
            source_text += (
                f"- 📄 **{source['source']}** — "
                f"Page {source['page']} "
                f"({source['policy_type']})\n"
            )

            seen.add(key)

    return source_text

In [ ]:
def chatbot(query):

    if not query.strip():
        return "Please enter a question."

    answer, sources = rag_answer(query)

    source_text = format_sources(sources)

    return answer + "\n" + source_text

In [ ]:
print(chatbot("What is the work from home policy?"))

In [ ]:
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Ask about Company Policies",
        placeholder="Example: How many casual leaves are allowed?"
    ),
    outputs=gr.Markdown(
        label="Policy Assistant"
    ),
    title="Company Policy RAG Assistant",
    description=(
        "Ask questions about company policies. "
        "Answers are generated only from the uploaded policy documents."
    ),
    examples=[
        "How many casual leaves are allowed?",
        "What is the work from home policy?",
        "What is the attendance requirement?",
        "What are the reimbursement rules?"
    ]
)

demo.launch(share=True)